# 3. Предразметка Grounding DINO
Модель предлагает рамки. Исполнитель исправляет границы, удаляет лишнее и добавляет пропущенные объекты.

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timedelta, timezone

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from seminar.data import read_json, write_json
from seminar.api import request, all_items, upload_tasks, classic_view, boxes_to_annotations

load_dotenv()
Path("results").mkdir(exist_ok=True)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from seminar.models import MODEL_ID, DINO_REVISION, PROMPTS, load_detector, detect
from seminar.metrics import select_predictions

images = read_json("data/dataset.json")
MODEL_ID, PROMPTS

### Предразметка
Готовые прогнозы уже есть. `RUN_DINO = True` пересчитает их на этих изображениях; дообучения нет.

In [ ]:
RUN_DINO = False

if RUN_DINO:
    processor, model = load_detector(device="cpu")
    predictions = {"config": {"model_id": MODEL_ID, "model_revision": DINO_REVISION, "prompts": PROMPTS}, "items": {}}
    for row in images:
        image = Image.open(row["image_path"]).convert("RGB")
        predictions["items"][row["image_id"]] = detect(image, processor, model)
        write_json("data/dino_predictions.json", predictions)
else:
    predictions = read_json("data/dino_predictions.json")

THRESHOLD = 0.25
predicted_boxes = {
    row["image_id"]: [d["box"] for d in select_predictions(predictions["items"][row["image_id"]], THRESHOLD)]
    for row in images
}

In [ ]:
example = images[0]
image = Image.open(example["image_path"])
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(image)
for x1, y1, x2, y2 in predicted_boxes[example["image_id"]]:
    ax.add_patch(Rectangle((x1 * image.width, y1 * image.height),
                          (x2 - x1) * image.width, (y2 - y1) * image.height,
                          fill=False, edgecolor="orange", linewidth=2))
ax.axis("off")
plt.show()

### Проект
Используем проект 10673 и пул 6756505. Для создания нового проекта и пула задайте оба `EXISTING_*_ID = None`.

In [ ]:
preview = [{"id": "0", "input_values": {
    "image": images[0]["image_url"],
    "image_id": images[0]["image_id"],
    "initial_boxes": boxes_to_annotations(predicted_boxes[images[0]["image_id"]]),
}}]
write_json("data/preview-dino.json", preview)
print(json.dumps(preview, ensure_ascii=False, indent=2))

In [ ]:
project_payload = {
    "public_name": "Выделите весь транспорт на фотографии [DINO]",
    "public_description": "Выделите объекты прямоугольниками. Если рамки уже есть, проверьте и исправьте их.",
    "public_instructions": Path("interface/instructions.html").read_text(encoding="utf-8"),
    "task_spec": {
        "input_spec": {
            "image": {"type": "url", "required": True},
            "image_id": {"type": "string", "required": True, "hidden": True},
            "initial_boxes": {"type": "json", "required": True, "hidden": False},
        },
        "output_spec": {
            "result": {"type": "json", "required": True},
            "preannotation_initialized": {"type": "boolean", "required": False},
        },
        "view_spec": classic_view("task"),
    },
    "assignments_issuing_type": "AUTOMATED",
    "assignments_automerge_enabled": False,
}

In [ ]:
EXISTING_PROJECT_ID = "10673"
EXISTING_POOL_ID = "6756505"

if EXISTING_POOL_ID and not EXISTING_PROJECT_ID:
    raise ValueError("Для существующего пула укажите EXISTING_PROJECT_ID")

project = (request("GET", f"projects/{EXISTING_PROJECT_ID}") if EXISTING_PROJECT_ID
           else request("POST", "projects", json=project_payload))
project_id = project["id"]
project_id


### Пул

In [ ]:
REWARD = 1
OVERLAP = 3

pool_payload = {
    "project_id": project_id,
    "private_name": "assisted",
    "may_contain_adult_content": False,
    "reward_per_assignment": REWARD,
    "assignment_max_duration_seconds": 1800,
    "will_expire": (datetime.now(timezone.utc) + timedelta(days=30)).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "auto_accept_solutions": True,
    "defaults": {"default_overlap_for_new_task_suites": OVERLAP},
    "mixer_config": {"real_tasks_count": 1, "golden_tasks_count": 0, "training_tasks_count": 0},
    "filter": {"and": [
        {"category": "computed", "key": "client_type", "operator": "EQ", "value": "BROWSER"},
        {"category": "profile", "key": "languages", "operator": "IN", "value": "RU"},
    ]},
}

In [ ]:
pool = (request("GET", f"pools/{EXISTING_POOL_ID}") if EXISTING_POOL_ID
        else request("POST", "pools", json=pool_payload))
assert str(pool["project_id"]) == str(project_id), "Пул принадлежит другому проекту"
pool_id = pool["id"]
REWARD = pool["reward_per_assignment"]
OVERLAP = pool["defaults"]["default_overlap_for_new_task_suites"]
pool_id


### Задания

In [ ]:
from seminar.results import annotation_results, workers_by_image

manual_file = Path("results/manual.json")
previous_workers = workers_by_image(annotation_results(manual_file, "manual")) if manual_file.exists() else {}
tasks = [
    {"pool_id": pool_id, "overlap": OVERLAP,
     "unavailable_for": previous_workers.get(row["image_id"], []),
     "input_values": {"image": row["image_url"], "image_id": row["image_id"],
                      "initial_boxes": boxes_to_annotations(predicted_boxes[row["image_id"]])}}
    for row in images
]
len(tasks), len(tasks) * OVERLAP * REWARD

In [ ]:
if EXISTING_POOL_ID:
    print("Задания уже в пуле:", len(all_items("tasks", pool_id=pool_id)))
else:
    upload_tasks(pool_id, tasks)


### Модерация

In [ ]:
validation_url = f"https://tasks.yandex.ru/api/new/requester/pools/{pool_id}/validate"
validation = request("GET", validation_url)
validation

Если активен `MUST_PASS_MODERATION`, отправим пул на модерацию.

In [ ]:
if EXISTING_POOL_ID:
    print("Этот пул уже отправлен на модерацию; статус — в предыдущей ячейке.")
elif any(item["type"] == "MUST_PASS_MODERATION" and item.get("active") for item in validation):
    moderation = request(
        "PUT",
        f"https://tasks.yandex.ru/api/new/requester/poolModeration/{pool_id}",
        json={"status": "READY"},
    )
    print(moderation)
else:
    print("Модерация не требуется или уже пройдена")

### Запуск
После прохождения модерации.

In [ ]:
validation = request("GET", validation_url)
blockers = [item for item in validation if item.get("active") and item.get("blocker", True)]
if blockers:
    raise ValueError(f"Пул пока нельзя открыть: {blockers}")
request("POST", f"pools/{pool_id}/open")

### Результаты
Выполните после разметки. Этот файл нужен четвёртому ноутбуку.

In [ ]:
assignments = all_items("assignments", pool_id=pool_id)
write_json("results/assisted.json", assignments)
pd.Series([a["status"] for a in assignments]).value_counts()